In [73]:
import re
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

# PART 1 - TASK 1

In [74]:
df = pd.read_csv('amazon.csv')

In [75]:
print("Shape: ",df.shape, "\nColumns: ",list(df.columns))

Shape:  (1465, 16) 
Columns:  ['product_id', 'product_name', 'category', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count', 'about_product', 'user_id', 'user_name', 'review_id', 'review_title', 'review_content', 'img_link', 'product_link']


In [76]:
display(df.head())

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53%,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61%,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


In [77]:
df.isnull().sum()

,0
product_id,0
product_name,0
category,0
discounted_price,0
actual_price,0
discount_percentage,0
rating,0
rating_count,2
about_product,0
user_id,0


In [78]:
df.dropna(inplace=True)

In [79]:
# duplicate data
df.duplicated().sum()

np.int64(0)

In [80]:
df['product_name'].duplicated().sum()

np.int64(128)

In [81]:
df['product_id'].duplicated().sum()

np.int64(114)

In [82]:
df['category'].str.split('|').str[0].value_counts()

,count
category,
Electronics,526
Computers&Accessories,451
Home&Kitchen,448
OfficeProducts,31
MusicalInstruments,2
HomeImprovement,2
Toys&Games,1
Car&Motorbike,1
Health&PersonalCare,1


# TASK 2

In [83]:
print("length of df before preprocessing: ",len(df))

length of df before preprocessing:  1463


In [84]:
df = df.drop_duplicates(subset='product_id')

In [85]:
df = df.drop_duplicates(subset='product_name')

In [86]:
print("length of df after removing duplicates: ",len(df))

length of df after removing duplicates:  1335


In [87]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1335 entries, 0 to 1464
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   product_id           1335 non-null   object
 1   product_name         1335 non-null   object
 2   category             1335 non-null   object
 3   discounted_price     1335 non-null   object
 4   actual_price         1335 non-null   object
 5   discount_percentage  1335 non-null   object
 6   rating               1335 non-null   object
 7   rating_count         1335 non-null   object
 8   about_product        1335 non-null   object
 9   user_id              1335 non-null   object
 10  user_name            1335 non-null   object
 11  review_id            1335 non-null   object
 12  review_title         1335 non-null   object
 13  review_content       1335 non-null   object
 14  img_link             1335 non-null   object
 15  product_link         1335 non-null   object
dtypes: object(1

In [88]:
df['rating'] = df['rating'].astype(dtype='float16')

ValueError: could not convert string to float: '|'

In [89]:
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

In [90]:
df['rating']=df['rating'].astype(dtype='float32')

In [91]:
df['raw_text'] = df['product_name'] + ' ' + df['category'].str.replace('|',' ', regex=False) + ' ' + df['about_product']

In [92]:
print(df['raw_text'][:5])

0    Wayona Nylon Braided USB to Lightning Fast Cha...
1    Ambrane Unbreakable 60W / 3A Fast Charging 1.5...
2    Sounce Fast Phone Charging Cable & Data Sync U...
3    boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...
4    Portronics Konnect L 1.2M Fast Charging 3A 8 P...
Name: raw_text, dtype: object


In [93]:
stop_words = ENGLISH_STOP_WORDS

def clean_text(text):
  text = str(text).lower()
  text = re.sub(r"^a-z0-9\s",' ', text)
  words = [w for w in text.split() if w not in stop_words] #removing stopwords
  return ' '.join(words)

In [94]:
df['clean_text'] = df['raw_text'].apply(clean_text)

In [95]:
print("\nRAW  :", df["raw_text"][0][:250])
print("CLEAN:", df["clean_text"][0][:250])


RAW  : Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey) Computers&Accessories Accessories&Peripherals Cables&Accessories Cables USBCables High 
CLEAN: wayona nylon braided usb lightning fast charging data sync cable compatible iphone 13, 12,11, x, 8, 7, 6, 5, ipad air, pro, mini (3 ft pack 1, grey) computers&accessories accessories&peripherals cables&accessories cables usbcables high compatibility 


# PART 2 - TASK 3

In [96]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(df['clean_text'])

print("TF_IDF matrix shape :", tfidf_matrix.shape)

TF_IDF matrix shape : (1335, 5000)


In [97]:
zeros = 1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])
print(f"sparsity: {zeros:.1%} zeros")

sparsity: 98.1% zeros


In [98]:
terms = tfidf.get_feature_names_out()
row = tfidf_matrix[0].toarray().ravel()
top = row.argsort()[::-1][:8]
print("\ntop words of the first product:")
print(pd.DataFrame({"term": terms[top], "tfidf": row[top].round(3)}).to_string(index=False))


top words of the first product:
       term  tfidf
     iphone  0.289
plus iphone  0.278
iphone plus  0.183
       ipad  0.165
       plus  0.148
      nylon  0.130
      12 11  0.123
       sync  0.116


# TASK 4

In [99]:
sim = cosine_similarity(tfidf_matrix)

print("similarity matrix shape: ", sim.shape)
print(f"size in memory: {sim.nbytes/1e6:.1f} MB")
print("product with itself (should be 1) ", round(sim[0,0],3))


similarity matrix shape:  (1335, 1335)
size in memory: 14.3 MB
product with itself (should be 1)  1.0


In [100]:
joblib.dump(sim.astype("float32"), 'similarity_matrix.joblib', compress=3)

['similarity_matrix.joblib']

## Why cosine?
- it checks the ANGLE between two vectors, not the length, so a product with a long description and one with a short description can still be "similar"
- tfidf values are never negative so the score is 0 (nothing common) to 1 (same)
- works fine on sparse text vectors


# PART 3 - TASK 5

In [101]:
def recommend(item_name, top_n = 5):
  # find index of prod
  matches = df.index[df['product_name']==item_name]
  if len(matches) == 0:
    print(f"{item_name} not found, please check entered name..")
    return None
  idx = matches[0]

  # similarity of prod
  scores = sim[idx]

  # sort high to low
  order = scores.argsort()[::-1]
  order = order[order != idx][:top_n]

  result = pd.DataFrame({
      "product_name": df["product_name"].iloc[order].str[:75].values,
      "similarity": scores[order].round(3)
  })

  result.index = range(1, len(result)+1)
  return result

In [102]:
# names are very long so will try to search using small names

def find_name(keyword):
  found = df.loc[df['product_name'].str.contains(keyword, regex = False), 'product_name']
  if len(found) == 0:
    return None
  return found.iloc[0]


In [103]:
try:
  for key in ['Prestige', 'Noise', 'BoAt', 'Havells Instanio', 'Fire-Bolt Ninja']:
    name = find_name(key)
    if name is None:
      print("\n",key, " not found\n")
      continue

    print("\nYou picked", name[:90])
    result = recommend(name)
    if result is not None:
      print("\n",result.to_string())
except Exception as e:
  print("\n\nError occured: ",e)



You picked Prestige 1.5 Litre Kettle 1500-watts, Red

                                                                   product_name  similarity
1  Karcher WD3 EU Wet and Dry Vacuum Cleaner, 1000 Watts Powerful Suction, 17        0.618
2  Eureka Forbes Active Clean 700 Watts Powerful Suction & Blower Vacuum Clean       0.584
3  Inalsa Vacuum Cleaner Wet and Dry Micro WD10 with 3in1 Multifunction Wet/Dr       0.579
4  AMERICAN MICRONIC- Imported Wet & Dry Vacuum Cleaner, 21 Litre Stainless St       0.522
5  Eureka Forbes Trendy Zip 1000 Watts powerful suction vacuum cleaner with re       0.520

You picked Noise Pulse Go Buzz Smart Watch Bluetooth Calling with 1.69" Display, 550 NITS, 150+ Cloud

                                                                   product_name  similarity
1  Samsung Galaxy M04 Light Green, 4GB RAM, 64GB Storage | Upto 8GB RAM with R       0.990
2  Samsung Galaxy M04 Dark Blue, 4GB RAM, 128GB Storage | Upto 8GB RAM with RA       0.978
3  Samsung Galaxy M1

In [104]:
recommend("Some Product That Does Not Exist")

Some Product That Does Not Exist not found, please check entered name..


In [105]:
idx = df.index[df["product_name"] == find_name("Prestige")][0]
one_row = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).ravel()
print("one-row method == full matrix:", np.allclose(one_row, sim[idx]))

one-row method == full matrix: True


In [106]:
df["short_category"] = df["category"].str.split("|").str[-1]      # last part, e.g. USBCables
df[["product_name", "short_category", "discounted_price", "rating",
    "product_link", "clean_text"]].to_csv("products_clean.csv", index=False)
print("saved products_clean.csv ->", df.shape[0], "products")

saved products_clean.csv -> 1335 products
